# Transformer Model 
The simple neural model from the previous step did not outperform the TF-IDF baseline. 
In this notebook I will shift from training a model to implement a pre-trained **Transformer Model**. The goal is to evaluate the perfomance of this advanced architecture through transfer learning. 
The model will be used directly for inference to see how its generalized language understanding compares to the previous trained models.

## Initial Setup 

In [10]:
# Installation of the necessary libraries
!pip install -q transformers torch

### Load of the data 
Transformer models are powerfull and are designed work best on natural, unprocessed text. 
this models derive value from the full context of the sentence. Therefore stop words, punctuation and capitalization will kept in the data, as the provide valuable signals for the model.

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load of the original dataset
file_path = '../data/IMDB Dataset.csv'
df = pd.read_csv(file_path)

# Define features (x) and target (y)
x = df['review'] 
y = df['sentiment']

# Split of the data into training/testing to maintain a consistent evaluation process
# In this notebook only will be use the x_test and y_test set.
x_train, x_test , y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)


## Transformer Pipeline
I will deploy a pre-trained pipeline from the Hugging Face 'transformers' library. This pipeline uses a model that has already been fine-tuned for sentiment analysis.

In [3]:
from transformers import pipeline 

#load a pre-trained sentiment analysis model from Hugging Face
print('loading sentiment analysis pipeline...')
sentiment_pipeline = pipeline('sentiment-analysis',model="distilbert-base-uncased-finetuned-sst-2-english")
print("Pipeline loaded")

loading sentiment analysis pipeline...


Device set to use cpu


Pipeline loaded


## Make predictions on the Test Set
The pipelines output requires minor formatting before the evaluation.  
It returns a list of dictionaries, each containing a label ('POSITIVE') and a confidence score. Our dataset labels are lowercase ('positive'). Therefore, a simple list comprehension is used to extract and convert the predicted labels to lowercase for a direct comparison.

In [4]:
# Convert the test set to a list for the pipeline
reviews_to_test = x_test.tolist()

# Run the pipeline on the 200 test reviews. 
# Need to add truncation=True to handle long reviews
print("Making predictions with the transformer model")
results = sentiment_pipeline(reviews_to_test, truncation=True)

# The pipeline outputs labels like 'POSITIVE' or 'NEGATIVE'. Dataset uses 'positive', let's convert them.
y_pred_transformer = [result['label'].lower() for result in results]
print('Predictions Complete.')



Making predictions with the transformer model
Predictions Complete.


## Evaluation of the Transformer model 

In [5]:
# The final report for the Transformer
print('\n Transformer Model Performance')
print(classification_report(y_test, y_pred_transformer))


 Transformer Model Performance
              precision    recall  f1-score   support

    negative       0.87      0.92      0.89      5000
    positive       0.92      0.86      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



# Final Project Results (Full Dataset)

## Model Performance Comparison  
This table compares the weighted average F-1 score of the three models developed in this project.

| Model                       | F1-Score (Weighted Avg) |
| --------------------------- | ----------------------- |
| **Baseline (TF-IDF + N-grams)** | **0.89**                    |
| Simple NN (Word Vectors + NN)  | 0.77                    |
| **Transformer** | **0.89** |

## Final Conclusion
The initial baseline model using TF-IDF and N-grams provided a strong starting point. While the simple neural model using spaCy's word vectors did not yield a significant improvent when deployed on the full dataset. 

The **pre-trained Transformer model** from Hugging Face achieved an excellent final F1-score of **0.89**. However the **TF-IDF + N-gram baseline model** when trained on the full dataset achieved the exact same F1-score **0.89**.

For a task like this one of binary sentiment analysis, a classic model could be just as powerful as a complex deep learning architecture. It demostrates that the N-gram feature approach was highly effective, and that the simple and faster baseline was a very efficient and high performing  solution for the specific task. 

## Interactive Demo
The final key decision of the project was deploying the transformer model for the interactive demonstration, even if the TF-IDF N-gram (baseline model) achieved the same performance. The limitation of the baseline model resides that its knowledge is limited to the n-grams it observed during the training. The transformer model instead is pre-trained on a massive and diverse corpus of text, giving it a more general understanding of the English language. The transformer model in the end provides the more robust and scalable solution, to handle the user-generated reviews.

This interactive demo of the transformer model, allows every person without need of specific knowledge can write his own review and retrieve an analyzis of the sentiment. For this we will use the library Gradio .

In [6]:
!pip install -q gradio


In [7]:
import gradio as gr

# This will be the function that configures the app
def predict_sentiment(text):
    """
    Takes a raw text string and returns a dictionary of the predicted label and its score
    """
    result = sentiment_pipeline(text, truncation=True)[0]
    label = result['label']
    score = result ['score']
    return {label: score}

# Creation of the gradio interface
iface = gr.Interface(fn=predict_sentiment,
                     inputs=gr.Textbox(lines=5, placeholder="Enter a movie review here..."),
                     outputs="label",
                     title="🎬 Movie Review Sentiment Analyzer",
                     description="Enter a movie review to see if the Transformer thinks it's POSITVE or NEGATIVE. This demo is powered by a DistilBERT model from Hugging Face.")

# Launch of the interface
iface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
